In [1]:
from collections import Counter, defaultdict
import numbers
from pathlib import Path
import os

import fitz
import numpy as np
import pandas as pd

schema_type = "CoreSchema"  # CoreSchema|OrganismTrends

# CoreSchema, dev
pdf_dir = "/ds/text/kiba-d/dev-set-100"
pdf_texts_file = "../data/processed/faktencheck/dev-set-100/predictions.jsonl"
reference_data = "../data/interim/faktencheck-db/faktenscheck_core_corrected.jsonl"

# CoreSchema, test
#pdf_dir = "/ds/text/kiba-d/splits/test"
#pdf_texts_file = "../data/processed/faktencheck/test/predictions.jsonl.gz"
#reference_data = "../data/interim/faktencheck-db/faktencheck-db-converted_2025-11-05.jsonl"

# OrganismTrends, dev
#pdf_dir = "/ds/text/kiba-d/dev-set-Wald-WVC"
#pdf_texts_file = "../data/processed/faktencheck/dev-set-Wald-WVC/predictions.jsonl.gz"
#reference_data = "../data/external/organism_trends/Referenz_Wald_korrigiert_angepasst.csv"

# OrganismTrends, test
#pdf_dir = "/ds/text/kiba-d/test-set-AuO-WVC"
#pdf_texts_file = "../data/processed/faktencheck/test-set-AuO-WVC/predictions.jsonl.gz"
#reference_data = "../data/external/organism_trends/Weighted Vote Count Agrar- und Offenland Literatur - Sheet1.csv"

In [2]:
def pdf_exists(filename):
    return (Path(pdf_dir) / f"{filename}.pdf").exists()


if schema_type == "CoreSchema":
    df = pd.read_json(reference_data, lines=True)
    df = df[["zotitem_ptr_id","biodiversity_level", "habitat", "ecosystem_type", "taxa"]]
    df = df[df["zotitem_ptr_id"].apply(pdf_exists)]
    
elif schema_type == "OrganismTrends":
    df = pd.read_csv(reference_data)
    df = df[df["Key"] != "#NV"]
    df = df[["Key","Antwortvariable", "Lebensraum", "Trend", "Hauptgruppe_RoteListen"]]
    df = df[df["Key"].apply(pdf_exists)]
    
else:
    print(f"Unsupported schema {schema_type}, expecting one of 'CoreSchema' or 'OrganismTrends'")

print(df.head())

             Key Antwortvariable Lebensraum     Trend Hauptgruppe_RoteListen
0       VJ4BEK2S        Abundanz       Wald  positive            Wirbeltiere
1       P2NC3SKD        Abundanz       Wald  positive               Pflanzen
2  Ellwanger2014        Abundanz       Wald  positive               Pflanzen
3       TYTG8U8R        Abundanz       Wald  negative            Wirbeltiere
5       SZ75F5Z9        Abundanz       Wald  positive            Wirbeltiere


In [3]:
counters = defaultdict(Counter)
totals = defaultdict(int)
#numeric_sums = defaultdict(float)

# count annotations
def process_variable(schema_variable, value):
    """Flattens lists/dicts and updates counters."""
    if value is None:
        return

    # dict → rekursiv flatten
    if isinstance(value, dict):
        for k, v in value.items():
            process_variable(f"{schema_variable}.{k}", v)
        return

    # list → jedes Element einzeln
    if isinstance(value, list):
        for item in value:
            process_variable(schema_variable, item)
        return

    # primitive value
    counters[schema_variable][str(value)] += 1
    totals[schema_variable] += 1

    #if isinstance(value, numbers.Number):
    #    numeric_sums[schema_variable] += value


for schema_variable in df.columns:
    for value in df[schema_variable]:
        process_variable(schema_variable, value)


# Output
print("\n=== Unique gold entries (support) in the reference data ===")
if schema_type == "CoreSchema":
    print(f"Unique docs with annotations: {df.notna().any(axis=1).sum()}")
else:
    print(f"Unique trend annotations: {df.notna().any(axis=1).sum()}")
    print(f"Unique docs with annotations: {len(df['Key'].unique())}")
for schema_variable in sorted(counters):
    print(f"\n=== {schema_variable} ===")
    print(f"Total values: {totals[schema_variable]}")

    #if numeric_sums[schema_variable] != 0:
    #    print(f"Numeric sum: {numeric_sums[schema_variable]}")

    for val, count in counters[schema_variable].most_common():
        print(f"  {val}: {count}")


=== Unique gold entries (support) in the reference data ===
Unique trend annotations: 202
Unique docs with annotations: 102

=== Antwortvariable ===
Total values: 202
  Abundanz: 126
  Artenzahl: 67
  ENS: 9

=== Hauptgruppe_RoteListen ===
Total values: 202
  Pflanzen: 107
  Wirbeltiere: 53
  Wirbellose: 30
  Pilze_Flechten: 12

=== Key ===
Total values: 202
  Meyer2021b: 12
  Günther2021: 9
  Schmidt2015: 5
  Schmidt2012: 5
  Heinrichs2014: 5
  ZU6JDWYE: 5
  Baumann2020: 5
  Fischer2009: 4
  Ahrns1998: 4
  AZDPE2RS: 4
  SVXF36H4: 4
  PUJAPGWQ: 4
  Heinrichs2012: 4
  Runge1981: 4
  9UA7424F: 3
  FFYFDKI9: 3
  52WFJFJS: 3
  3B3C8R6S: 3
  Kudernatsch2019: 3
  Schmidt2017: 3
  T2GFNI8R: 3
  Kolbe1992: 3
  EB8KV3G9: 3
  UEQEAG8X: 3
  vanRiesen2003: 3
  Westhus1990: 2
  Wuttky1982: 2
  FN348QUV: 2
  G7H698R6: 2
  Fischer2015B: 2
  BX6CS4BF: 2
  H7XXBFEY: 2
  JHMHRAMC: 2
  Schneider2021: 2
  Jöbges2006: 2
  3YVJV7GT: 2
  DSKU37ZR: 2
  Dierschke2013: 2
  PHEVUEKM: 2
  TPNQGJLF: 2
  Schmidt20

In [4]:
df = pd.read_json(pdf_texts_file, lines=True)
print(df.head())

      file_name                                               text  \
0  26QI8J7B.pdf  Ecological Complexity 7 (2010) 260–272\n\n\n[C...   
1  2AWXR7KG.pdf                                                      
2  2P53UVJA.pdf  ### **Invasion Ecology**\n\n\n# **Invasion Eco...   
3  2SNIVXFW.pdf  This is a pre-print version of the following p...   
4  2US2J9GI.pdf  Ecological Applications, 22(8), 2012, pp. 2065...   

   response_content  structured  structured_with_metadata  reasoning_content  \
0               NaN         NaN                       NaN                NaN   
1               NaN         NaN                       NaN                NaN   
2               NaN         NaN                       NaN                NaN   
3               NaN         NaN                       NaN                NaN   
4               NaN         NaN                       NaN                NaN   

   messages  messages_formatted errors errors_long  
0       NaN                 NaN     []       

In [5]:
def count_words(df, col):
    word_counts = []

    for value in df[col].dropna():
        if not isinstance(value, str):
            continue

        words = value.split()  # whitespace split
        word_counts.append(len(words))

    # Summary stats
    sum_words = np.sum(word_counts) if word_counts else 0
    avg_words = np.mean(word_counts) if word_counts else 0
    med_words = np.median(word_counts) if word_counts else 0
    max_words = max(word_counts) if word_counts else 0
    min_words = min(word_counts) if word_counts else 0

    print(f"Column: {col}")
    print(f"Average words per doc: {avg_words:.2f}")
    print(f"Median words per doc: {med_words}")
    print(f"Max words per doc: {max_words}")
    print(f"Min words per doc: {min_words}")
    print(f"Total words: {sum_words}")


col = "text"
count_words(df, col)

Column: text
Average words per doc: 14193.14
Median words per doc: 9045.0
Max words per doc: 519607
Min words per doc: 0
Total words: 5804996


In [6]:
def count_pages(base_dir):
    page_counts = []

    for file_path in os.listdir(base_dir):
        try:
            doc = fitz.open(Path(base_dir, file_path))
            num_pages = doc.page_count
            page_counts.append(num_pages)
            doc.close()

        except Exception as e:
            print(f"Error reading {file_path}: {e}")

    sum_pages = np.sum(page_counts) if page_counts else 0
    avg_pages = np.mean(page_counts) if page_counts else 0
    med_pages = np.median(page_counts) if page_counts else 0
    max_pages = max(page_counts) if page_counts else 0
    min_pages = min(page_counts) if page_counts else 0

    print(f"Average pages per PDF: {avg_pages:.2f}")
    print(f"Median pages per PDF: {med_pages}")
    print(f"Max pages: {max_pages}")
    print(f"Min pages: {min_pages}")
    print(f"Total pages: {sum_pages}")

count_pages(pdf_dir)

Average pages per PDF: 30.48
Median pages per PDF: 14.0
Max pages: 894
Min pages: 1
Total pages: 12468
